# MT Model Training with LoRA - Sequential Fine-Tuning Experiments

This notebook tests whether **sequential fine-tuning with LoRA adapters** (similar language → baseline) improves MT performance compared to **direct fine-tuning with LoRA** (baseline only).

## Why LoRA?
- **Memory Efficient:** Only trains low-rank adapter matrices instead of full model weights
- **Faster Training:** Fewer parameters to update = faster iterations
- **Same Hypothesis:** Test if similarity transfer helps when using parameter-efficient fine-tuning

## Experimental Design: Sequential Fine-Tuning with LoRA

For each target language, we create TWO models:

### 1. Baseline Models (Direct LoRA Training)
- **Start:** mBART-50 (pretrained)
- **Train:** Apply LoRA adapters and train Distant language → Target language (e.g., `en→tl`)
- **Result:** Baseline Model with LoRA adapters

### 2. Experimental Models (Sequential LoRA Fine-Tuning)
- **Start:** mBART-50 (pretrained)
- **Step 1:** Apply LoRA adapters and train Similar language → Target language (e.g., `bik→tl`)
- **Step 2:** Load Stage 1 LoRA adapters and **continue training** on Distant language → Target language (e.g., `en→tl`)
- **Result:** Experimental Model (with similarity transfer via LoRA)

### Three Target Languages
1. **Tagalog (tl)**
   - Baseline: `en→tl` only
   - Experimental: `bik→tl` THEN `en→tl`

2. **Ilonggo/Hiligaynon (hil)**
   - Baseline: `en→hil` only
   - Experimental: `msb→hil` THEN `en→hil`

3. **Waray (war)**
   - Baseline: `en→war` only
   - Experimental: `hil→war` THEN `en→war`

## Research Question
**Does "warming up" the model with LoRA on a similar low-resource language first improve performance on the baseline task?**

Expected: `BLEU(Experimental) > BLEU(Baseline)`

## Install Required Packages

Run this cell first if packages are not installed. Note the addition of `peft` for LoRA support.

In [ ]:
# Uncomment and run if needed
# !pip install transformers datasets evaluate sacrebleu torch sentencepiece accelerate peft

## Imports

In [1]:
from transformers import (
    MBartForConditionalGeneration,
    MBartTokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import Dataset, DatasetDict
import evaluate
import numpy as np
import torch
from pathlib import Path
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3050


## Configuration

Set up training configurations for all language pairs, including LoRA-specific parameters.

In [2]:
# Model configuration
MODEL_NAME = "facebook/mbart-large-50"
MAX_LENGTH = 128
BATCH_SIZE = 4  # Reduced for LoRA (still memory efficient)
LEARNING_RATE = 3e-4  # Higher LR often works well with LoRA
NUM_EPOCHS_STAGE1 = 3  # For similar language training (Stage 1)
NUM_EPOCHS_STAGE2 = 3  # For baseline training (Stage 2)

# LoRA configuration
LORA_R = 8  # Rank of LoRA matrices
LORA_ALPHA = 16  # Scaling factor
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # Apply LoRA to attention layers

# Directories
DATA_DIR = Path("../data/splits")
OUTPUT_DIR = Path("../models")
LOGS_DIR = Path("../logs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Experimental configurations
EXPERIMENTS = {
    "tagalog": {
        "target": "tl",
        "baseline_pair": "en-tl",
        "similar_pair": "bik-tl",
        "baseline_config": {
            "src_lang": "en_XX",
            "tgt_lang": "tl_XX",
        },
        "similar_config": {
            "src_lang": "tl_XX",
            "tgt_lang": "tl_XX",
        }
    },
    "ilonggo": {
        "target": "hil",
        "baseline_pair": "en-hil",
        "similar_pair": "msb-hil",
        "baseline_config": {
            "src_lang": "en_XX",
            "tgt_lang": "tl_XX",
        },
        "similar_config": {
            "src_lang": "tl_XX",
            "tgt_lang": "tl_XX",
        }
    },
    "waray": {
        "target": "war",
        "baseline_pair": "en-war",
        "similar_pair": "hil-war",
        "baseline_config": {
            "src_lang": "en_XX",
            "tgt_lang": "tl_XX",
        },
        "similar_config": {
            "src_lang": "tl_XX",
            "tgt_lang": "tl_XX",
        }
    }
}

print("Experimental Configuration (LoRA-based):")
print(f"  Model: {MODEL_NAME}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  LoRA rank (r): {LORA_R}")
print(f"  LoRA alpha: {LORA_ALPHA}")
print(f"  LoRA target modules: {LORA_TARGET_MODULES}")
print(f"  Epochs (Stage 1): {NUM_EPOCHS_STAGE1}")
print(f"  Epochs (Stage 2): {NUM_EPOCHS_STAGE2}")
print(f"\nTarget Languages: {len(EXPERIMENTS)}")
for lang, config in EXPERIMENTS.items():
    print(f"  - {lang.capitalize()}: {config['baseline_pair']} (baseline) vs {config['similar_pair']}→{config['baseline_pair']} (sequential)")

Experimental Configuration (LoRA-based):
  Model: facebook/mbart-large-50
  Batch size: 4
  Learning rate: 0.0003
  LoRA rank (r): 8
  LoRA alpha: 16
  LoRA target modules: ['q_proj', 'v_proj']
  Epochs (Stage 1): 3
  Epochs (Stage 2): 3

Target Languages: 3
  - Tagalog: en-tl (baseline) vs bik-tl→en-tl (sequential)
  - Ilonggo: en-hil (baseline) vs msb-hil→en-hil (sequential)
  - Waray: en-war (baseline) vs hil-war→en-war (sequential)


## Helper Functions

Functions to load data, preprocess, evaluate models, and apply LoRA adapters.

In [3]:
def load_data_for_pair(pair_name):
    """Load train and dev splits for a language pair."""
    src_code, tgt_code = pair_name.split("-")
    pair_dir = DATA_DIR / pair_name
    
    with open(pair_dir / f"train.{src_code}", "r", encoding="utf-8") as f:
        train_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"train.{tgt_code}", "r", encoding="utf-8") as f:
        train_tgt = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"dev.{src_code}", "r", encoding="utf-8") as f:
        dev_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"dev.{tgt_code}", "r", encoding="utf-8") as f:
        dev_tgt = [line.strip() for line in f.readlines()]
    
    train_dataset = Dataset.from_dict({"src": train_src, "tgt": train_tgt})
    dev_dataset = Dataset.from_dict({"src": dev_src, "tgt": dev_tgt})
    
    dataset_dict = DatasetDict({"train": train_dataset, "validation": dev_dataset})
    
    print(f"Loaded {pair_name}: Train={len(train_dataset)}, Dev={len(dev_dataset)}")
    return dataset_dict


def create_preprocess_function(tokenizer, src_lang, tgt_lang, max_length):
    """Create preprocessing function for tokenization."""
    def preprocess(batch):
        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            batch["src"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )
        tokenizer.tgt_lang = tgt_lang
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                batch["tgt"],
                truncation=True,
                padding="max_length",
                max_length=max_length
            )
        inputs["labels"] = labels["input_ids"]
        return inputs
    return preprocess


def create_compute_metrics(tokenizer):
    """Create function to compute BLEU score during evaluation."""
    bleu = evaluate.load("sacrebleu")
    
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        if isinstance(preds, tuple):
            preds = preds[0]
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        result = bleu.compute(
            predictions=decoded_preds,
            references=[[label] for label in decoded_labels]
        )
        return {"bleu": result["score"]}
    return compute_metrics


def apply_lora_to_model(model):
    """Apply LoRA adapters to the model."""
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="SEQ_2_SEQ_LM",
    )
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()
    return peft_model


print("✓ Helper functions defined")

✓ Helper functions defined


## Training Functions

Functions to train single stages and run complete experiments with LoRA.

In [ ]:
def train_single_stage(pair_name, config, model_name_or_path, output_subdir, num_epochs, stage_name="", is_stage2=False):
    """Train a single stage with LoRA adapters."""
    print("\n" + "="*80)
    print(f"Training: {pair_name.upper()}")
    if stage_name:
        print(f"Stage: {stage_name}")
    print("="*80)
    
    # Load tokenizer
    print("\n1. Loading tokenizer...")
    tokenizer = MBartTokenizerFast.from_pretrained(MODEL_NAME)
    
    # Load model
    print("\n2. Loading model...")
    if is_stage2:
        # Stage 2: Load base model then merge and reload adapters from Stage 1
        print(f"   Loading Stage 1 adapters from: {model_name_or_path}")
        base_model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
        model = PeftModel.from_pretrained(base_model, model_name_or_path)
        print("   Stage 1 adapters loaded successfully")
    else:
        # Stage 1 or Baseline: Load base model and apply new LoRA
        print(f"   Loading base model: {MODEL_NAME}")
        base_model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
        model = apply_lora_to_model(base_model)
    
    # Load dataset
    print("\n3. Loading dataset...")
    dataset = load_data_for_pair(pair_name)
    
    # Tokenize
    print("\n4. Tokenizing dataset...")
    preprocess_fn = create_preprocess_function(
        tokenizer, config['src_lang'], config['tgt_lang'], MAX_LENGTH
    )
    tokenized_dataset = dataset.map(preprocess_fn, batched=True)
    
    # Training arguments
    output_dir = OUTPUT_DIR / output_subdir
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=num_epochs,
        save_strategy="epoch",
        save_total_limit=2,
        predict_with_generate=True,
        logging_dir=str(LOGS_DIR / output_subdir),
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    
    # Create trainer
    print("\n5. Setting up trainer...")
    compute_metrics_fn = create_compute_metrics(tokenizer)
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_fn,
    )
    
    # Train
    print("\n6. Starting training...")
    train_result = trainer.train()
    
    # Save model (LoRA adapters)
    print("\n7. Saving LoRA adapters and tokenizer...")
    final_model_path = output_dir / "final_model"
    model.save_pretrained(str(final_model_path))
    tokenizer.save_pretrained(str(final_model_path))
    
    # Evaluate
    print("\n8. Final evaluation...")
    eval_results = trainer.evaluate()
    
    # Save results
    results = {
        "pair": pair_name,
        "stage": stage_name,
        "model_source": model_name_or_path,
        "lora_config": {
            "r": LORA_R,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "target_modules": LORA_TARGET_MODULES
        },
        "train_results": {
            "train_loss": train_result.training_loss,
            "train_runtime": train_result.metrics["train_runtime"],
        },
        "eval_results": eval_results
    }
    
    results_file = output_dir / "training_results.json"
    with open(results_file, "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✓ Training complete")
    print(f"  Final BLEU: {eval_results['eval_bleu']:.2f}")
    print(f"  LoRA adapters saved to: {final_model_path}")
    
    return results, str(final_model_path)


def train_experiment(target_lang_name, experiment_config):
    """Run complete experiment for one target language with LoRA."""
    print("\n" + "#"*80)
    print(f"# EXPERIMENT (LoRA): {target_lang_name.upper()}")
    print("#"*80)
    
    baseline_pair = experiment_config['baseline_pair']
    similar_pair = experiment_config['similar_pair']
    all_results = {}
    
    # BASELINE: Direct LoRA training on distant → target
    print(f"\n{'='*80}")
    print(f"BASELINE (LoRA): {baseline_pair}")
    print("="*80)
    baseline_results, baseline_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_baseline_lora",
        num_epochs=NUM_EPOCHS_STAGE2,
        stage_name="Baseline (Direct LoRA)",
        is_stage2=False
    )
    all_results['baseline'] = baseline_results
    
    # EXPERIMENTAL - STAGE 1: LoRA on similar → target
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL (LoRA) - STAGE 1: {similar_pair}")
    print("="*80)
    stage1_results, stage1_model_path = train_single_stage(
        pair_name=similar_pair,
        config=experiment_config['similar_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_experimental_stage1_lora",
        num_epochs=NUM_EPOCHS_STAGE1,
        stage_name="Stage 1 (Similar Language LoRA)",
        is_stage2=False
    )
    all_results['experimental_stage1'] = stage1_results
    
    # EXPERIMENTAL - STAGE 2: Continue LoRA from Stage 1 on distant → target
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL (LoRA) - STAGE 2: {baseline_pair}")
    print(f"Continuing from Stage 1 LoRA adapters")
    print("="*80)
    stage2_results, stage2_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=stage1_model_path,
        output_subdir=f"{target_lang_name}_experimental_stage2_lora",
        num_epochs=NUM_EPOCHS_STAGE2,
        stage_name="Stage 2 (Baseline after Similar LoRA)",
        is_stage2=True
    )
    all_results['experimental_stage2'] = stage2_results
    
    # SUMMARY
    print("\n" + "="*80)
    print(f"EXPERIMENT COMPLETE (LoRA): {target_lang_name.upper()}")
    print("="*80)
    print(f"\nBaseline (LoRA): {all_results['baseline']['eval_results']['eval_bleu']:.2f} BLEU")
    print(f"Experimental Stage 1: {all_results['experimental_stage1']['eval_results']['eval_bleu']:.2f} BLEU")
    print(f"Experimental Stage 2: {all_results['experimental_stage2']['eval_results']['eval_bleu']:.2f} BLEU")
    
    improvement = all_results['experimental_stage2']['eval_results']['eval_bleu'] - all_results['baseline']['eval_results']['eval_bleu']
    print(f"\nImprovement: {improvement:+.2f} BLEU points")
    if improvement > 0:
        print("✓ Sequential LoRA fine-tuning IMPROVED performance")
    elif improvement < 0:
        print("✗ Sequential LoRA fine-tuning DEGRADED performance")
    else:
        print("= No difference in performance")
    
    # Save summary
    summary_file = OUTPUT_DIR / f"{target_lang_name}_lora_experiment_summary.json"
    with open(summary_file, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSummary saved to: {summary_file}")
    
    return all_results


print("✓ Training functions defined")

✓ Training functions defined


## Run Single Experiment (LoRA)

Test with one target language first to verify the LoRA setup.

In [6]:
# Run one experiment (uncomment to test)
target_lang = "tagalog"
results = train_experiment(target_lang, EXPERIMENTS[target_lang])


################################################################################
# EXPERIMENT (LoRA): TAGALOG
################################################################################

BASELINE (LoRA): en-tl

Training: EN-TL
Stage: Baseline (Direct LoRA)

1. Loading tokenizer...


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizerFast'.



2. Loading model...
   Loading base model: facebook/mbart-large-50
trainable params: 1,179,648 || all params: 612,059,136 || trainable%: 0.1927

3. Loading dataset...
Loaded en-tl: Train=2358, Dev=294

4. Tokenizing dataset...


Map:   0%|          | 0/2358 [00:00<?, ? examples/s]

Map:   0%|          | 0/294 [00:00<?, ? examples/s]

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

## Run All Experiments (LoRA)

Run all three experiments with LoRA adapters. **WARNING: This will take several hours!**

For each target language:
1. Train baseline LoRA model (distant → target)
2. Train Stage 1 LoRA (similar → target)
3. Train Stage 2 LoRA (continue with distant → target)

Total: 9 training runs (3 per target language × 3 target languages)

In [ ]:
# Run all experiments (uncomment to run)
# all_experiment_results = {}
#
# for target_lang, exp_config in EXPERIMENTS.items():
#     try:
#         results = train_experiment(target_lang, exp_config)
#         all_experiment_results[target_lang] = results
#     except Exception as e:
#         print(f"\n❌ Error in {target_lang} experiment: {e}")
#         import traceback
#         traceback.print_exc()
#         continue
#
# # Save overall summary
# final_summary_file = OUTPUT_DIR / "all_lora_experiments_summary.json"
# with open(final_summary_file, "w") as f:
#     json.dump(all_experiment_results, f, indent=2)
#
# print("\n" + "#"*80)
# print("# ALL LoRA EXPERIMENTS COMPLETE")
# print("#"*80)
# print(f"\nFinal summary saved to: {final_summary_file}")
#
# # Print comparison table
# print("\n" + "="*80)
# print("RESULTS SUMMARY (LoRA)")
# print("="*80)
# print(f"{'Target':<20} {'Baseline BLEU':<15} {'Sequential BLEU':<15} {'Improvement':<15}")
# print("-"*80)
# for target_lang, results in all_experiment_results.items():
#     baseline_bleu = results['baseline']['eval_results']['eval_bleu']
#     sequential_bleu = results['experimental_stage2']['eval_results']['eval_bleu']
#     improvement = sequential_bleu - baseline_bleu
#     print(f"{target_lang.capitalize():<20} {baseline_bleu:<15.2f} {sequential_bleu:<15.2f} {improvement:+.2f}")

## Summary and Next Steps

### What This Notebook Does

This notebook implements **sequential fine-tuning experiments with LoRA adapters** to test whether "warming up" a model on similar language data improves performance on the baseline task while using parameter-efficient fine-tuning.

### Key Differences from Full Fine-Tuning

1. **Memory Efficiency:** Only LoRA adapter weights are trained (~0.5-2% of model parameters)
2. **Faster Training:** Fewer parameters = faster iterations and lower GPU memory
3. **Same Hypothesis:** Does similarity transfer help when using PEFT?

### Output Structure
```
models/
  ├── tagalog_baseline_lora/
  │   └── final_model/          # Baseline: en→tl (LoRA only)
  ├── tagalog_experimental_stage1_lora/
  │   └── final_model/          # Stage 1: bik→tl (LoRA)
  ├── tagalog_experimental_stage2_lora/
  │   └── final_model/          # Stage 2: bik→tl THEN en→tl (LoRA)
  ├── tagalog_lora_experiment_summary.json
  │
  ├── ilonggo_baseline_lora/
  ├── ilonggo_experimental_stage1_lora/
  ├── ilonggo_experimental_stage2_lora/
  ├── ilonggo_lora_experiment_summary.json
  │
  ├── waray_baseline_lora/
  ├── waray_experimental_stage1_lora/
  ├── waray_experimental_stage2_lora/
  ├── waray_lora_experiment_summary.json
  │
  └── all_lora_experiments_summary.json
```

### Comparison with Full Fine-Tuning

After running both notebooks (`train-models.ipynb` and `train-models-lora.ipynb`), you can:
1. Compare BLEU scores: Full fine-tuning vs. LoRA
2. Compare training time and memory usage
3. Analyze if similarity transfer effect is similar with LoRA

### Expected Analysis

For each target language:
- Compare `BLEU(Baseline LoRA)` vs. `BLEU(Experimental LoRA)`
- Calculate improvement: `Δ BLEU = Experimental - Baseline`
- Compare with full fine-tuning results

### Training Notes

- **Total Training Runs:** 9 (3 per target language)
- **GPU Recommended:** Works on 4GB VRAM with batch_size=4
- **Time Estimate:** ~30-45 min per run = 4-7 hours total
- **Memory:** Much lower than full fine-tuning (~2-4GB vs 8-12GB)

### Tips

1. Start with one experiment to verify setup
2. Monitor GPU memory with `nvidia-smi`
3. Increase batch size if you have more VRAM
4. LoRA adapters are small (~10-50MB) vs full models (~2GB)
5. You can merge LoRA adapters with base model later if needed